# MEPI Figure 5 — final-test prediction and residual diagnostics

## Chạy thủ công trong VS Code

1. Chọn kernel **Python (trans-core)**.
2. Chọn **Restart Kernel**.
3. Bấm **Run All**.

Notebook tự tìm thư mục gốc dự án dù VS Code khởi chạy từ thư mục gốc hay từ `notebooks/`. Nó chỉ đọc CSV dự đoán final-test đã đóng băng và ghi đè đúng hai tệp hình Figure 5; không tải checkpoint, không huấn luyện và không truy cập lại test loader.

This notebook performs **post-hoc visualization only** using the already frozen MEPI v1.5 final-test prediction CSV. It does not train or select a model, tune hyperparameters, load or change a checkpoint, alter any split, modify predictions or residuals, filter or aggregate samples, remove outliers, or recalibrate uncertainty.

All 90 final-test measurements are shown. Historical `core_loss` column names are displayed using the corrected manuscript term **residual loss, P_loss**, in watts. LSP is treated as a dimensionless constructed relative thermal-stress proxy—not measured lifetime, remaining useful life, time to failure, or measured degradation. The plots make no claim that residuals are random, predictions are unbiased, uncertainty is calibrated, the samples represent 90 independent operating conditions, or the model generalizes beyond the investigated operating domain.

In [ ]:
from pathlib import Path
import hashlib
import json
import subprocess

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

SOURCE_RELATIVE = Path("reports/MEPI_V1_5_POSTHOC_TEST_PREDICTIONS.csv")
METRICS_RELATIVE = Path("reports/MEPI_V1_5_POSTHOC_TEST_METRICS.json")
OUTPUT_DIR_RELATIVE = Path("reports/figures")

current = Path.cwd().resolve()
PROJECT_ROOT = next(
    (candidate for candidate in (current, *current.parents) if (candidate / SOURCE_RELATIVE).is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not locate {SOURCE_RELATIVE}; run this notebook from the project or notebooks directory."
    )

SOURCE_CSV = PROJECT_ROOT / SOURCE_RELATIVE
METRICS_JSON = PROJECT_ROOT / METRICS_RELATIVE
OUTPUT_DIR = PROJECT_ROOT / OUTPUT_DIR_RELATIVE
PNG_PATH = OUTPUT_DIR / "MEPI_Fig5_final_test_diagnostics.png"
PDF_PATH = OUTPUT_DIR / "MEPI_Fig5_final_test_diagnostics.pdf"

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

print(f"Project root: {PROJECT_ROOT}")
print(f"Frozen source: {SOURCE_CSV}")

In [ ]:
source_sha256 = sha256_file(SOURCE_CSV)
data = pd.read_csv(SOURCE_CSV)

if METRICS_JSON.is_file():
    posthoc_record = json.loads(METRICS_JSON.read_text(encoding="utf-8"))
    recorded_sha256 = posthoc_record.get("predictions_sha256")
    if recorded_sha256 and source_sha256 != recorded_sha256:
        raise AssertionError(
            "Frozen prediction CSV SHA-256 does not match the existing MEPI v1.5 post-hoc record: "
            f"observed={source_sha256}, recorded={recorded_sha256}"
        )

print(f"Loaded {len(data)} rows from the frozen prediction artifact.")
print(f"Source SHA-256: {source_sha256}")

In [ ]:
REQUIRED_COLUMNS = [
    "core_id",
    "efficiency_measured_percent",
    "efficiency_predicted_percent",
    "efficiency_residual_percent",
    "core_loss_measured_w",
    "core_loss_predicted_w",
    "core_loss_residual_w",
    "lsp_measured",
    "lsp_predicted",
    "lsp_residual",
]
PLOTTED_COLUMNS = [column for column in REQUIRED_COLUMNS if column != "core_id"]

missing_columns = sorted(set(REQUIRED_COLUMNS) - set(data.columns))
if missing_columns:
    raise AssertionError(f"Missing required frozen-prediction columns: {missing_columns}")
if len(data) != 90:
    raise AssertionError(f"Expected exactly 90 final-test rows; found {len(data)}.")
if "split" in data.columns:
    split_values = set(data["split"].dropna().astype(str).str.strip().str.lower())
    if split_values != {"test"} or data["split"].isna().any():
        raise AssertionError(f"Expected every row to have split='test'; found {sorted(split_values)}.")
if data[PLOTTED_COLUMNS].isna().any().any():
    affected = data[PLOTTED_COLUMNS].columns[data[PLOTTED_COLUMNS].isna().any()].tolist()
    raise AssertionError(f"Missing values found in plotted columns: {affected}")
if not np.isfinite(data[PLOTTED_COLUMNS].to_numpy(dtype=float)).all():
    raise AssertionError("Non-finite values found in plotted columns.")
if "condition_group_id" in data.columns:
    group_count = data["condition_group_id"].nunique(dropna=True)
    if data["condition_group_id"].isna().any() or group_count != 9:
        raise AssertionError(
            f"Expected 9 non-missing final-test condition groups; found {group_count}."
        )

print("Input integrity checks: PASS")

In [ ]:
RESIDUAL_CHECKS = {
    "efficiency_residual_percent": ("efficiency_predicted_percent", "efficiency_measured_percent"),
    "core_loss_residual_w": ("core_loss_predicted_w", "core_loss_measured_w"),
    "lsp_residual": ("lsp_predicted", "lsp_measured"),
}
RESIDUAL_ATOL = 1e-12
RESIDUAL_RTOL = 1e-10

for stored_column, (predicted_column, reference_column) in RESIDUAL_CHECKS.items():
    calculated = data[predicted_column].to_numpy() - data[reference_column].to_numpy()
    stored = data[stored_column].to_numpy()
    if not np.allclose(calculated, stored, rtol=RESIDUAL_RTOL, atol=RESIDUAL_ATOL):
        maximum_difference = float(np.max(np.abs(calculated - stored)))
        raise AssertionError(
            f"Stored residual mismatch in {stored_column}; max absolute discrepancy={maximum_difference:.3e}."
        )

print("Residual identities (prediction - reference): PASS")

In [ ]:
core_labels = data["core_id"].astype(str).str.strip().str.upper()
condition_groups = data["condition_group_id"].nunique() if "condition_group_id" in data.columns else "not available"

print("Integrity summary")
print(f"  Rows: {len(data)}")
print(f"  Unique condition groups: {condition_groups}")
print(f"  FE samples: {(core_labels == 'FE').sum()}")
print(f"  Commercial samples: {(core_labels == 'COMMERCIAL').sum()}")
for label, column, unit in [
    ("Efficiency", "efficiency_measured_percent", "%"),
    ("Residual loss, P_loss", "core_loss_measured_w", "W"),
    ("LSP", "lsp_measured", ""),
]:
    suffix = f" {unit}" if unit else ""
    print(f"  Reference {label}: {data[column].min():.6g} to {data[column].max():.6g}{suffix}")

In [ ]:
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Arial", "Liberation Sans"],
    "font.size": 9.5,
    "axes.labelsize": 10,
    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,
    "axes.linewidth": 1.0,
    "xtick.major.width": 1.0,
    "ytick.major.width": 1.0,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

fig, axes = plt.subplots(2, 3, figsize=(12, 7), constrained_layout=True, facecolor="white")
point_style = dict(s=28, alpha=0.9, color="#1900FF", edgecolors="white", linewidths=0.65)
line_style = dict(color="#080808", linewidth=1.5, linestyle=(0, (5, 3)), zorder=0)

targets = [
    {
        "reference": "efficiency_measured_percent",
        "prediction": "efficiency_predicted_percent",
        "residual": "efficiency_residual_percent",
        "reference_label": "Reference efficiency (%)",
        "prediction_label": "Predicted efficiency (%)",
        "residual_label": "Residual (pp)",
    },
    {
        "reference": "core_loss_measured_w",
        "prediction": "core_loss_predicted_w",
        "residual": "core_loss_residual_w",
        "reference_label": "Reference residual loss, P_loss (W)",
        "prediction_label": "Predicted residual loss, P_loss (W)",
        "residual_label": "Residual (W)",
    },
    {
        "reference": "lsp_measured",
        "prediction": "lsp_predicted",
        "residual": "lsp_residual",
        "reference_label": "Reference LSP",
        "prediction_label": "Predicted LSP",
        "residual_label": "Residual",
    },
]
panel_labels = (("(a)", "(b)", "(c)"), ("(d)", "(e)", "(f)"))
plotted_counts = []

for column, target in enumerate(targets):
    reference = data[target["reference"]].to_numpy()
    prediction = data[target["prediction"]].to_numpy()
    residual = data[target["residual"]].to_numpy()

    top_axis = axes[0, column]
    top_scatter = top_axis.scatter(reference, prediction, **point_style)
    combined_min = float(min(reference.min(), prediction.min()))
    combined_max = float(max(reference.max(), prediction.max()))
    combined_span = combined_max - combined_min
    margin = 0.05 * combined_span if combined_span > 0 else max(abs(combined_min) * 0.05, 1e-6)
    limits = (combined_min - margin, combined_max + margin)
    top_axis.plot(limits, limits, **line_style)
    top_axis.set_xlim(limits)
    top_axis.set_ylim(limits)
    top_axis.set_aspect("equal", adjustable="box")
    top_axis.set_xlabel(target["reference_label"])
    top_axis.set_ylabel(target["prediction_label"])

    bottom_axis = axes[1, column]
    bottom_scatter = bottom_axis.scatter(reference, residual, **point_style)
    bottom_axis.axhline(0.0, **line_style)
    bottom_axis.set_xlabel(target["reference_label"])
    bottom_axis.set_ylabel(target["residual_label"])
    bottom_axis.margins(x=0.05, y=0.10)

    plotted_counts.extend([len(top_scatter.get_offsets()), len(bottom_scatter.get_offsets())])

for row in range(2):
    for column in range(3):
        axis = axes[row, column]
        axis.set_facecolor("white")
        axis.grid(False)
        axis.tick_params(direction="out", length=3.5, width=0.8)
        axis.text(
            0.02, 0.98, panel_labels[row][column],
            transform=axis.transAxes, ha="left", va="top",
            fontsize=11, fontweight="bold",
        )
        for spine in axis.spines.values():
            spine.set_linewidth(0.8)

if plotted_counts != [90] * 6:
    raise AssertionError(f"Expected 90 plotted points in every panel; found {plotted_counts}.")

print(f"Plotted sample counts by panel: {plotted_counts}")

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(PNG_PATH, dpi=400, facecolor="white", bbox_inches="tight")
fig.savefig(
    PDF_PATH,
    format="pdf",
    facecolor="white",
    bbox_inches="tight",
    metadata={"Creator": "MEPI Figure 5 post-hoc notebook", "CreationDate": None, "ModDate": None},
)

if not PNG_PATH.is_file() or not PDF_PATH.is_file():
    raise RuntimeError("Figure export failed: one or more output files are missing.")

print(f"Saved PNG: {PNG_PATH}")
print(f"Saved PDF: {PDF_PATH}")

In [ ]:
display(fig)
plt.close(fig)

In [ ]:
def current_git_commit(project_root):
    try:
        completed = subprocess.run(
            ["git", "-C", str(project_root), "rev-parse", "HEAD"],
            check=True, capture_output=True, text=True, timeout=5,
        )
        return completed.stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError, subprocess.TimeoutExpired):
        return "UNAVAILABLE (not a Git working tree or git command failed)"

print("Provenance summary")
print(f"  Source CSV path: {SOURCE_CSV}")
print(f"  Source CSV SHA-256: {source_sha256}")
print(f"  PNG SHA-256: {sha256_file(PNG_PATH)}")
print(f"  PDF SHA-256: {sha256_file(PDF_PATH)}")
print(f"  Plotted sample count: {len(data)} in each of 6 panels")
print(f"  Current git commit SHA: {current_git_commit(PROJECT_ROOT)}")

**Figure 5.** Final-test prediction and residual diagnostics of MEPI for (a, d) transformer efficiency, (b, e) residual loss P_loss, and (c, f) LSP. Panels (a–c) compare reference and predicted values, with the dashed diagonal indicating ideal one-to-one agreement. Panels (d–f) show the corresponding prediction residuals, defined as prediction minus reference value, with the dashed horizontal line indicating zero residual.